# Data Quality — Manage Checks Notebook

Use this notebook as the single place to add new checks and update existing rows in `check_registry`.

Identity key for add/update rows: `(check_name, workspace_id, dataset_id)`.

In [ ]:
# Configuration
# Load shared settings from config.py in Lakehouse.
import sys
sys.path.append("/lakehouse/default/Files/code")

try:
    from config import LAKEHOUSE_NAME, SCHEMA_NAME, MAX_RETRY_ATTEMPTS, INITIAL_RETRY_DELAY, DAX_TIMEOUT_SECONDS, RUN_TABLE_MAINTENANCE, MAINTENANCE_DAY_UTC
except ImportError:
    LAKEHOUSE_NAME = "MyLakehouse"
    SCHEMA_NAME = "data_quality"
    MAX_RETRY_ATTEMPTS = 3
    INITIAL_RETRY_DELAY = 2
    DAX_TIMEOUT_SECONDS = 60
    RUN_TABLE_MAINTENANCE = False
    MAINTENANCE_DAY_UTC = 6

FULL_SCHEMA = f"{LAKEHOUSE_NAME}.{SCHEMA_NAME}"
print(f"Schema: {FULL_SCHEMA}")

In [ ]:
from IPython.display import display

display(spark.sql(f"""
SELECT
    r.check_name,
    r.model_name,
    r.workspace_id,
    r.dataset_id,
    r.workspace_name,
    r.dataset_name,
    r.run_frequency,
    r.fail_delta_pct_threshold,
    r.is_active,
    r.is_baseline,
    COALESCE(c.baseline_mode, 'MODEL') AS baseline_mode,
    c.static_baseline_value
FROM {FULL_SCHEMA}.check_registry r
LEFT JOIN {FULL_SCHEMA}.check_baseline_config c
  ON r.check_name = c.check_name
ORDER BY r.check_name, r.model_name, r.workspace_id, r.dataset_id
""").toPandas())

## How To Specify Changes

Single change tuple:
`(check_name, workspace_id, dataset_id, model_name, workspace_name, dataset_name, dax_expression, run_frequency, fail_delta_pct_threshold, is_active, is_baseline, baseline_mode, static_baseline_value)`

Rules:
- First 3 fields are required identity selectors
- Existing rows: use `None` to keep a row field unchanged
- New rows: provide non-empty values for `model_name`, `workspace_name`, `dataset_name`, `dax_expression`, `run_frequency`, and `fail_delta_pct_threshold`
- When an active row's model metadata, DAX, frequency, or threshold changes, the prior active row is retired (`is_active = false`) and a new active version is inserted
- `dax_expression` is still required for runnable rows even when `baseline_mode = STATIC`; STATIC only replaces the baseline comparison value
- `run_frequency` (when provided) must be `ONCE_PER_DAY` or `MULTIPLE_PER_DAY`
- `fail_delta_pct_threshold` is stored as a percent value, so `0.30` means `0.30%` and `30.0` means `30%`
- `fail_delta_pct_threshold` (when provided) must be numeric and `>= 0`
- `is_baseline = True` designates this row as baseline for MODEL-mode checks and clears baseline from other active rows for that check
- Clearing the current baseline (`is_baseline = False`) is rejected unless another row in the same submission is promoted to baseline for MODEL-mode checks
- `baseline_mode` is optional and applies at the `check_name` level
- `baseline_mode` options are `MODEL` or `STATIC`
- `STATIC` mode requires `static_baseline_value` to be numeric
- For a baseline-mode-only change, use any existing row for that check as the selector and leave row fields as `None`
- If a check has exactly one active row and baseline mode is `MODEL`, that row is auto-forced to baseline

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, DoubleType
from pyspark.sql.functions import trim, upper
from IPython.display import display

# -- EDIT BELOW ---------------------------------------------------------------
changes = [
    # Example: update an existing row OR add a new row by providing a new identity key
    (
        "Total Sales",                                     # check_name                 - required
        "66666666-7777-8888-9999-000000000000",            # workspace_id               - required
        "ffffffff-1111-2222-3333-444444444444",            # dataset_id                 - required
        "Sales AMER v2",                                   # model_name                 - None to keep unchanged
        None,                                               # workspace_name             - None to keep unchanged
        None,                                               # dataset_name               - None to keep unchanged
        'EVALUATE ROW("value", [Total Revenue])',          # dax_expression             - still required for runnable rows, even when baseline_mode = STATIC
        None,                                               # run_frequency              - None to keep unchanged
        0.30,                                               # fail_delta_pct_threshold   - percent value; 0.30 means 0.30%, not 30%
        None,                                               # is_active                  - None to keep unchanged
        None,                                               # is_baseline                - True to promote baseline, False to clear (guarded), None to keep unchanged
        None,                                               # baseline_mode              - None to keep unchanged, or MODEL / STATIC
        None,                                               # static_baseline_value      - required when baseline_mode = STATIC
    ),
    # Example: baseline-mode-only change using any existing row selector for the check
    # (
    #     "Total Sales",
    #     "66666666-7777-8888-9999-000000000000",
    #     "ffffffff-1111-2222-3333-444444444444",
    #     None,
    #     None,
    #     None,
    #     None,                                           # dax_expression - leave as None only when the existing row already has a valid DAX
    #     None,
    #     None,
    #     None,
    #     None,
    #     "STATIC",
    #     1250000.0,
    # ),
]
# -----------------------------------------------------------------------------

if len(changes) == 0:
    raise ValueError("No changes provided. Add at least one row to changes.")

change_schema = StructType([
    StructField("check_name",                StringType(),  False),
    StructField("workspace_id",              StringType(),  False),
    StructField("dataset_id",                StringType(),  False),
    StructField("model_name",                StringType(),  True),
    StructField("workspace_name",            StringType(),  True),
    StructField("dataset_name",              StringType(),  True),
    StructField("dax_expression",            StringType(),  True),
    StructField("run_frequency",             StringType(),  True),
    StructField("fail_delta_pct_threshold",  DoubleType(),  True),
    StructField("is_active",                 BooleanType(), True),
    StructField("is_baseline",               BooleanType(), True),
    StructField("baseline_mode",             StringType(),  True),
    StructField("static_baseline_value",     DoubleType(),  True),
])

changes_df = spark.createDataFrame(changes, schema=change_schema)
changes_df = (
    changes_df.withColumn("check_name", trim("check_name"))
              .withColumn("workspace_id", trim("workspace_id"))
              .withColumn("dataset_id", trim("dataset_id"))
              .withColumn("model_name", trim("model_name"))
              .withColumn("workspace_name", trim("workspace_name"))
              .withColumn("dataset_name", trim("dataset_name"))
              .withColumn("dax_expression", trim("dax_expression"))
              .withColumn("run_frequency", upper(trim("run_frequency")))
              .withColumn("baseline_mode", upper(trim("baseline_mode")))
)
changes_df.createOrReplaceTempView("_changes")

active_registry_df = spark.sql(f"""
    SELECT
        check_name,
        model_name,
        workspace_id,
        dataset_id,
        workspace_name,
        dataset_name,
        dax_expression,
        run_frequency,
        fail_delta_pct_threshold,
        is_active,
        is_baseline
    FROM {FULL_SCHEMA}.check_registry
    WHERE is_active = true
""")
active_registry_df.createOrReplaceTempView("_active_registry")

# Keep baseline config synchronized for any existing checks.
spark.sql(f"""
MERGE INTO {FULL_SCHEMA}.check_baseline_config c
USING (
    SELECT DISTINCT check_name
    FROM {FULL_SCHEMA}.check_registry
) s
ON c.check_name = s.check_name
WHEN NOT MATCHED THEN INSERT (check_name, baseline_mode, static_baseline_value, updated_at)
VALUES (s.check_name, 'MODEL', NULL, current_timestamp())
""")

updated_selector_count = 0
updated_mode_count = 0

u = changes_df.select(
    "check_name",
    "workspace_id",
    "dataset_id",
    "model_name",
    "workspace_name",
    "dataset_name",
    "dax_expression",
    "run_frequency",
    "fail_delta_pct_threshold",
    "is_active",
    "is_baseline",
)
u.createOrReplaceTempView("_updates")

bad_selector_rows = u.filter(
    "check_name IS NULL OR check_name = '' OR workspace_id IS NULL OR workspace_id = '' OR dataset_id IS NULL OR dataset_id = ''"
).toPandas()
if len(bad_selector_rows) > 0:
    raise ValueError("Change rows must include non-blank check_name/workspace_id/dataset_id.")

dupe_selectors = u.groupBy("check_name", "workspace_id", "dataset_id").count().filter("count > 1").toPandas()
if len(dupe_selectors) > 0:
    raise ValueError("Duplicate change selectors found in submission.")

bad_frequency_rows = u.filter(
    "run_frequency IS NOT NULL AND run_frequency <> '' AND run_frequency NOT IN ('ONCE_PER_DAY', 'MULTIPLE_PER_DAY')"
).toPandas()
if len(bad_frequency_rows) > 0:
    raise ValueError("Invalid run_frequency in changes. Allowed: ONCE_PER_DAY, MULTIPLE_PER_DAY")

bad_threshold_rows = u.filter(
    "fail_delta_pct_threshold IS NOT NULL AND fail_delta_pct_threshold < 0"
).toPandas()
if len(bad_threshold_rows) > 0:
    raise ValueError("Invalid fail_delta_pct_threshold in changes. Value must be >= 0.")

# Validate required fields for rows that will be newly inserted.
insert_missing_required = spark.sql(f"""
    SELECT
        u.check_name, u.workspace_id, u.dataset_id,
        u.model_name, u.workspace_name, u.dataset_name, u.run_frequency, u.fail_delta_pct_threshold
    FROM _updates u
    LEFT JOIN _active_registry t
      ON u.check_name = t.check_name
     AND u.workspace_id = t.workspace_id
     AND u.dataset_id = t.dataset_id
    WHERE t.check_name IS NULL
      AND (
          u.model_name IS NULL OR u.model_name = ''
          OR u.workspace_name IS NULL OR u.workspace_name = ''
          OR u.dataset_name IS NULL OR u.dataset_name = ''
          OR u.dax_expression IS NULL OR u.dax_expression = ''
          OR u.run_frequency IS NULL OR u.run_frequency = ''
          OR u.fail_delta_pct_threshold IS NULL
      )
""").toPandas()
if len(insert_missing_required) > 0:
    raise ValueError(
        "New rows require model_name/workspace_name/dataset_name/dax_expression/run_frequency/fail_delta_pct_threshold. Missing fields for:\n"
        + insert_missing_required.head(20).to_string(index=False)
    )

# Prevent multiple promotions in one submission for the same check_name.
baseline_promotions = u.filter("is_baseline = true").groupBy("check_name").count().filter("count > 1").toPandas()
if len(baseline_promotions) > 0:
    raise ValueError(
        "Submission promotes multiple baseline rows for the same check_name. "
        "Provide at most one is_baseline=True row per check_name."
    )

# Baseline rows must stay active after update.
inactive_promotion = spark.sql(f"""
    SELECT u.check_name, u.workspace_id, u.dataset_id
    FROM _updates u
    INNER JOIN _active_registry t
      ON u.check_name = t.check_name
     AND u.workspace_id = t.workspace_id
     AND u.dataset_id = t.dataset_id
    WHERE u.is_baseline = true
      AND COALESCE(u.is_active, t.is_active) = false
""").toPandas()
if len(inactive_promotion) > 0:
    raise ValueError("A baseline row must be active. Set is_active=True or remove is_baseline=True for these changes.")

# Reject clearing the current baseline unless the same submission promotes a replacement baseline.
clearing_without_replacement = spark.sql(f"""
    SELECT DISTINCT t.check_name
    FROM _updates u
    INNER JOIN _active_registry t
      ON u.check_name = t.check_name
     AND u.workspace_id = t.workspace_id
     AND u.dataset_id = t.dataset_id
    LEFT JOIN {FULL_SCHEMA}.check_baseline_config c
      ON t.check_name = c.check_name
    WHERE u.is_baseline = false
      AND t.is_baseline = true
      AND t.is_active = true
      AND UPPER(COALESCE(c.baseline_mode, 'MODEL')) = 'MODEL'
      AND NOT EXISTS (
          SELECT 1
          FROM _updates u2
          WHERE u2.check_name = t.check_name
            AND u2.is_baseline = true
      )
""").toPandas()
if len(clearing_without_replacement) > 0:
    projected_active_counts = spark.sql(f"""
        WITH projected AS (
            SELECT
                t.check_name,
                CASE
                    WHEN u.is_active IS NULL THEN t.is_active
                    ELSE u.is_active
                END AS projected_is_active
            FROM _active_registry t
            LEFT JOIN _updates u
              ON u.check_name = t.check_name
             AND u.workspace_id = t.workspace_id
             AND u.dataset_id = t.dataset_id
        )
        SELECT
            check_name,
            SUM(CASE WHEN projected_is_active = true THEN 1 ELSE 0 END) AS projected_active_count
        FROM projected
        GROUP BY check_name
    """).toPandas()

    blocked_checks = clearing_without_replacement.merge(
        projected_active_counts,
        on="check_name",
        how="left"
    )
    blocked_checks = blocked_checks[blocked_checks["projected_active_count"] > 0]
    if len(blocked_checks) > 0:
        raise ValueError(
            "Cannot clear the current baseline without assigning a new one when active rows remain. "
            "Use is_baseline=True on the replacement row in the same submission."
        )

change_plan_df = spark.sql("""
    SELECT
        u.check_name,
        u.workspace_id,
        u.dataset_id,
        u.model_name,
        u.workspace_name,
        u.dataset_name,
        u.dax_expression,
        u.run_frequency,
        u.fail_delta_pct_threshold,
        u.is_active,
        u.is_baseline,
        CASE WHEN t.check_name IS NOT NULL THEN true ELSE false END AS matched_active_row,
        CASE
            WHEN t.check_name IS NOT NULL
             AND COALESCE(u.is_active, t.is_active, true) = true
             AND (
                 (u.model_name IS NOT NULL AND NOT (u.model_name <=> t.model_name))
                 OR (u.workspace_name IS NOT NULL AND NOT (u.workspace_name <=> t.workspace_name))
                 OR (u.dataset_name IS NOT NULL AND NOT (u.dataset_name <=> t.dataset_name))
                 OR (u.dax_expression IS NOT NULL AND NOT (u.dax_expression <=> t.dax_expression))
                 OR (u.run_frequency IS NOT NULL AND NOT ((CASE WHEN u.run_frequency = '' THEN NULL ELSE u.run_frequency END) <=> t.run_frequency))
                 OR (u.fail_delta_pct_threshold IS NOT NULL AND NOT (u.fail_delta_pct_threshold <=> t.fail_delta_pct_threshold))
             )
            THEN true
            ELSE false
        END AS requires_new_version,
        COALESCE(u.model_name, t.model_name) AS final_model_name,
        COALESCE(u.workspace_name, t.workspace_name) AS final_workspace_name,
        COALESCE(u.dataset_name, t.dataset_name) AS final_dataset_name,
        COALESCE(u.dax_expression, t.dax_expression) AS final_dax_expression,
        COALESCE(CASE WHEN u.run_frequency = '' THEN NULL ELSE u.run_frequency END, t.run_frequency) AS final_run_frequency,
        COALESCE(u.fail_delta_pct_threshold, t.fail_delta_pct_threshold) AS final_fail_delta_pct_threshold,
        COALESCE(u.is_active, t.is_active, true) AS final_is_active,
        COALESCE(u.is_baseline, t.is_baseline, false) AS final_is_baseline
    FROM _updates u
    LEFT JOIN _active_registry t
      ON u.check_name = t.check_name
     AND u.workspace_id = t.workspace_id
     AND u.dataset_id = t.dataset_id
""")
change_plan_df.createOrReplaceTempView("_change_plan")

versioned_selector_count = change_plan_df.filter(
    "matched_active_row = true AND requires_new_version = true"
).count()
in_place_selector_count = change_plan_df.filter(
    "matched_active_row = true AND requires_new_version = false"
).count()
insert_selector_count = change_plan_df.filter(
    "matched_active_row = false OR requires_new_version = true"
).count()

if versioned_selector_count > 0:
    spark.sql(f"""
        MERGE INTO {FULL_SCHEMA}.check_registry t
        USING (
            SELECT DISTINCT check_name, workspace_id, dataset_id
            FROM _change_plan
            WHERE matched_active_row = true
              AND requires_new_version = true
        ) s
        ON t.check_name = s.check_name
       AND t.workspace_id = s.workspace_id
       AND t.dataset_id = s.dataset_id
       AND t.is_active = true
        WHEN MATCHED THEN UPDATE SET
          t.is_active = false,
          t.is_baseline = false
    """)

if in_place_selector_count > 0:
    spark.sql("""
        CREATE OR REPLACE TEMP VIEW _direct_updates AS
        SELECT
            u.check_name,
            u.workspace_id,
            u.dataset_id,
            u.model_name,
            u.workspace_name,
            u.dataset_name,
            u.dax_expression,
            u.run_frequency,
            u.fail_delta_pct_threshold,
            u.is_active,
            u.is_baseline
        FROM _updates u
        INNER JOIN _change_plan p
          ON u.check_name = p.check_name
         AND u.workspace_id = p.workspace_id
         AND u.dataset_id = p.dataset_id
        WHERE p.matched_active_row = true
          AND p.requires_new_version = false
    """)

    spark.sql(f"""
        MERGE INTO {FULL_SCHEMA}.check_registry t
        USING _direct_updates s
          ON t.check_name = s.check_name
         AND t.workspace_id = s.workspace_id
         AND t.dataset_id = s.dataset_id
         AND t.is_active = true
        WHEN MATCHED THEN UPDATE SET
          t.model_name                = COALESCE(s.model_name, t.model_name),
          t.workspace_name            = COALESCE(s.workspace_name, t.workspace_name),
          t.dataset_name              = COALESCE(s.dataset_name, t.dataset_name),
          t.dax_expression            = COALESCE(s.dax_expression, t.dax_expression),
          t.run_frequency             = COALESCE(CASE WHEN s.run_frequency = '' THEN NULL ELSE s.run_frequency END, t.run_frequency),
          t.fail_delta_pct_threshold  = COALESCE(s.fail_delta_pct_threshold, t.fail_delta_pct_threshold),
          t.is_active                 = COALESCE(s.is_active, t.is_active),
          t.is_baseline               = COALESCE(s.is_baseline, t.is_baseline)
    """)

if insert_selector_count > 0:
    spark.sql(f"""
        INSERT INTO {FULL_SCHEMA}.check_registry (
            check_name, model_name, workspace_id, dataset_id, workspace_name, dataset_name,
            dax_expression, run_frequency, fail_delta_pct_threshold, is_active, is_baseline
        )
        SELECT
            check_name,
            final_model_name,
            workspace_id,
            dataset_id,
            final_workspace_name,
            final_dataset_name,
            final_dax_expression,
            final_run_frequency,
            final_fail_delta_pct_threshold,
            COALESCE(final_is_active, true),
            COALESCE(final_is_baseline, false)
        FROM _change_plan
        WHERE matched_active_row = false
           OR requires_new_version = true
    """)

# When a row is promoted to baseline, clear the flag on all other active rows for that check_name.
newly_baseline_df = spark.sql(f"""
    SELECT t.check_name, t.workspace_id, t.dataset_id
    FROM {FULL_SCHEMA}.check_registry t
    INNER JOIN _updates u
      ON t.check_name = u.check_name
     AND t.workspace_id = u.workspace_id
     AND t.dataset_id = u.dataset_id
    LEFT JOIN {FULL_SCHEMA}.check_baseline_config c
      ON t.check_name = c.check_name
    WHERE t.is_baseline = true
      AND t.is_active = true
      AND u.is_baseline = true
      AND UPPER(COALESCE(c.baseline_mode, 'MODEL')) = 'MODEL'
""").toPandas()

for _, row in newly_baseline_df.iterrows():
    cn = row["check_name"].replace("'", "''")
    wid = row["workspace_id"].replace("'", "''")
    did = row["dataset_id"].replace("'", "''")
    spark.sql(f"""
        UPDATE {FULL_SCHEMA}.check_registry
        SET is_baseline = false
        WHERE check_name = '{cn}'
          AND is_active = true
          AND NOT (workspace_id = '{wid}' AND dataset_id = '{did}')
    """)
    print(f"  Transferred baseline for '{row['check_name']}' -> workspace={row['workspace_id']} / dataset={row['dataset_id']}")

updated_selector_count = len(changes)

mode_df = changes_df.filter(
    "baseline_mode IS NOT NULL OR static_baseline_value IS NOT NULL"
).select("check_name", "baseline_mode", "static_baseline_value")
mode_df.createOrReplaceTempView("_baseline_mode_updates")

if len([row for row in changes if row[11] is not None or row[12] is not None]) > 0:
    bad_mode_selectors = mode_df.filter("check_name IS NULL OR check_name = ''").toPandas()
    if len(bad_mode_selectors) > 0:
        raise ValueError("baseline_mode changes require non-blank check_name values.")

    bad_modes = mode_df.filter("baseline_mode IS NULL OR baseline_mode NOT IN ('MODEL', 'STATIC')").toPandas()
    if len(bad_modes) > 0:
        raise ValueError("Invalid baseline_mode in changes. Allowed values: MODEL, STATIC")

    static_missing = mode_df.filter("baseline_mode = 'STATIC' AND static_baseline_value IS NULL").toPandas()
    if len(static_missing) > 0:
        raise ValueError("STATIC baseline_mode requires static_baseline_value to be provided.")

    mode_dupes = mode_df.groupBy("check_name").count().filter("count > 1").toPandas()
    if len(mode_dupes) > 0:
        raise ValueError("Provide baseline_mode at most once per check_name in the changes list.")

    missing_mode_targets = spark.sql(f"""
        SELECT m.check_name
        FROM _baseline_mode_updates m
        LEFT JOIN (
            SELECT DISTINCT check_name
            FROM {FULL_SCHEMA}.check_registry
        ) r
          ON m.check_name = r.check_name
        WHERE r.check_name IS NULL
    """).toPandas()
    if len(missing_mode_targets) > 0:
        raise RuntimeError("One or more baseline_mode change check_name values do not exist in check_registry.")

    spark.sql(f"""
        MERGE INTO {FULL_SCHEMA}.check_baseline_config c
        USING _baseline_mode_updates s
          ON c.check_name = s.check_name
        WHEN MATCHED THEN UPDATE SET
          c.baseline_mode = s.baseline_mode,
          c.static_baseline_value = CASE WHEN s.baseline_mode = 'STATIC' THEN s.static_baseline_value ELSE NULL END,
          c.updated_at = current_timestamp()
        WHEN NOT MATCHED THEN INSERT (check_name, baseline_mode, static_baseline_value, updated_at)
        VALUES (
            s.check_name,
            s.baseline_mode,
            CASE WHEN s.baseline_mode = 'STATIC' THEN s.static_baseline_value ELSE NULL END,
            current_timestamp()
        )
    """)

    # Optional hygiene: when switching to STATIC, clear model baseline flags for active rows.
    static_checks = spark.sql("""
        SELECT check_name
        FROM _baseline_mode_updates
        WHERE baseline_mode = 'STATIC'
    """).toPandas()
    for _, row in static_checks.iterrows():
        cn = row["check_name"].replace("'", "''")
        spark.sql(f"""
            UPDATE {FULL_SCHEMA}.check_registry
            SET is_baseline = false
            WHERE check_name = '{cn}'
              AND is_active = true
        """)

    updated_mode_count = len([row for row in changes if row[11] is not None or row[12] is not None])

# Force policy: if a MODEL-mode check has exactly one active row, that row must be baseline.
# Delta does not support subqueries in UPDATE, so collect target names first.
_single_active_checks = spark.sql(f"""
    SELECT r.check_name
    FROM {FULL_SCHEMA}.check_registry r
    LEFT JOIN {FULL_SCHEMA}.check_baseline_config c
      ON r.check_name = c.check_name
    WHERE r.is_active = true
      AND UPPER(COALESCE(c.baseline_mode, 'MODEL')) = 'MODEL'
    GROUP BY r.check_name
    HAVING COUNT(*) = 1
""").toPandas()["check_name"].tolist()
if _single_active_checks:
    _escaped = [n.replace("'", "''") for n in _single_active_checks]
    _name_list = ", ".join(f"'{n}'" for n in _escaped)
    spark.sql(f"""
        UPDATE {FULL_SCHEMA}.check_registry
        SET is_baseline = true
        WHERE is_active = true
          AND check_name IN ({_name_list})
    """)

# Enforce strict baseline invariant for active MODEL checks only.
active_baseline_issues = spark.sql(f"""
    SELECT
        r.check_name,
        c.baseline_mode,
        COUNT(*) AS active_row_count,
        SUM(CASE WHEN r.is_baseline = true THEN 1 ELSE 0 END) AS active_baseline_count
    FROM {FULL_SCHEMA}.check_registry r
    LEFT JOIN {FULL_SCHEMA}.check_baseline_config c
      ON r.check_name = c.check_name
    WHERE r.is_active = true
    GROUP BY r.check_name, c.baseline_mode
    HAVING UPPER(COALESCE(c.baseline_mode, 'MODEL')) = 'MODEL'
       AND SUM(CASE WHEN r.is_baseline = true THEN 1 ELSE 0 END) <> 1
""").toPandas()
if len(active_baseline_issues) > 0:
    raise RuntimeError(
        "Baseline invariant violation after update: each active MODEL check_name must have exactly one active baseline row.\n"
        + active_baseline_issues.head(20).to_string(index=False)
    )

# Validate STATIC mode has a numeric baseline value.
invalid_static_config = spark.sql(f"""
    SELECT c.check_name
    FROM {FULL_SCHEMA}.check_baseline_config c
    INNER JOIN (
        SELECT DISTINCT check_name
        FROM {FULL_SCHEMA}.check_registry
        WHERE is_active = true
    ) a
      ON c.check_name = a.check_name
    WHERE UPPER(COALESCE(c.baseline_mode, 'MODEL')) = 'STATIC'
      AND c.static_baseline_value IS NULL
""").toPandas()
if len(invalid_static_config) > 0:
    raise RuntimeError(
        "STATIC baseline mode requires static_baseline_value for active checks. Missing value for check_name(s): "
        + ", ".join(invalid_static_config["check_name"].tolist())
    )

invalid_threshold_rows = spark.sql(f"""
    SELECT check_name, model_name, workspace_id, dataset_id, fail_delta_pct_threshold
    FROM {FULL_SCHEMA}.check_registry
    WHERE is_active = true
      AND (fail_delta_pct_threshold IS NULL OR fail_delta_pct_threshold < 0)
""").toPandas()
if len(invalid_threshold_rows) > 0:
    raise RuntimeError(
        "Active rows must have a non-negative fail_delta_pct_threshold after update.\n"
        + invalid_threshold_rows.head(20).to_string(index=False)
    )

frequency_conflicts = spark.sql(f"""
    SELECT check_name
    FROM {FULL_SCHEMA}.check_registry
    WHERE is_active = true
    GROUP BY check_name
    HAVING COUNT(DISTINCT UPPER(TRIM(COALESCE(run_frequency, '')))) > 1
""").toPandas()
if len(frequency_conflicts) > 0:
    raise RuntimeError(
        "run_frequency consistency violation detected after update for check_name(s): "
        + ", ".join(frequency_conflicts["check_name"].tolist())
    )

print(f"Applied changes to {updated_selector_count} row selector(s).")
print(f"Retired {versioned_selector_count} prior active row(s).")
print(f"Applied baseline-mode changes to {updated_mode_count} check(s).")

# --- Verify current active rows for touched selectors ---
updated_rows = spark.sql(f"""
SELECT
    t.check_name, t.model_name, t.workspace_id, t.dataset_id,
    t.workspace_name, t.dataset_name, t.run_frequency, t.fail_delta_pct_threshold, t.is_active, t.is_baseline,
    COALESCE(c.baseline_mode, 'MODEL') AS baseline_mode,
    c.static_baseline_value
FROM {FULL_SCHEMA}.check_registry t
LEFT JOIN {FULL_SCHEMA}.check_baseline_config c
  ON t.check_name = c.check_name
INNER JOIN _updates u
  ON t.check_name = u.check_name
 AND t.workspace_id = u.workspace_id
 AND t.dataset_id = u.dataset_id
WHERE t.is_active = true
ORDER BY t.check_name, t.model_name, t.workspace_id, t.dataset_id
""").toPandas()

display(updated_rows)
print(f"\nVerified {len(updated_rows)} current active row(s).")

if updated_mode_count > 0:
    updated_modes = spark.sql(f"""
    SELECT check_name, baseline_mode, static_baseline_value, updated_at
    FROM {FULL_SCHEMA}.check_baseline_config
    WHERE check_name IN (
        SELECT check_name FROM _baseline_mode_updates
    )
    ORDER BY check_name
    """).toPandas()

    display(updated_modes)
    print(f"Verified {len(updated_modes)} baseline-mode change row(s).")